# ทำความสะอาดข้อมูล ขั้นที่ 3/5: ตรวจและจัดการค่าผิดปกติ (Outliers)

| ขั้น | เรื่อง | สถานะ |
|---|---|---|
| 1 | โครงสร้าง คีย์หลัก ข้อมูลซ้ำ ขอบเขต | ✅ |
| 2 | ค่าว่าง | ✅ |
| **3 (ไฟล์นี้)** | **ค่าผิดปกติ** | เกณฑ์ 3.5.1.2.2 (และใช้ต่อใน 3.5.2.3.1) |
| 4 | มาตรฐานรูปแบบ + ความสอดคล้องของตัวเลข | |
| 5 | Data Dictionary + ไฟล์สะอาดสุดท้าย | |

**หัวใจของขั้นนี้: ค่าที่ "สูง/ต่ำผิดปกติ" ไม่ได้แปลว่า "ผิด" เสมอไป**

ค่าผิดปกติมี 2 แบบ ซึ่งต้องจัดการต่างกัน

| แบบ | ตัวอย่าง | วิธีจัดการ |
|---|---|---|
| **ผิดจริง** (error) | สัดส่วนเกิน 100%, ค่าติดลบที่เป็นไปไม่ได้, ค่ากระโดดที่ไม่มีเหตุการณ์รองรับ | แก้ไข หรือเปลี่ยนเป็นค่าว่าง |
| **สูงจริง** (real extreme) | ไอซ์แลนด์ใช้ไฟต่อคนสูงที่สุดในโลก | **เก็บไว้** แต่ติดป้าย (flag) และเลือกสถิติ/กราฟที่ไม่ถูกดึงเพี้ยน |

ลำดับการตรวจ:
1. **กฎเชิงโดเมน** (ค่าที่เป็นไปไม่ได้)
2. **สถิติ IQR และ Z-score** (ค่าที่ไกลจากกลุ่ม)
3. **การกระโดดตามเวลา** (ค่าที่เปลี่ยนจากปีก่อนแรงผิดปกติ)
4. **ความแม่นยำของประเทศเล็กมาก** (ตัวเลขถูกปัดจนเพี้ยน)

**อินพุต:** `clean_output/clean_step2.csv` → **เอาต์พุต:** `clean_step3.csv`, `outlier_report.csv`, และเพิ่มบรรทัดใน `cleaning_log.csv`

> รัน **Kernel → Restart Kernel and Run All Cells** (ต้องรันขั้นที่ 2 สำเร็จก่อน)

## 3.1 โหลดผลจากขั้นที่ 2

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

OUT_DIR = "clean_output"
IN_FILE = os.path.join(OUT_DIR, "clean_step2.csv")
LOG_FILE = os.path.join(OUT_DIR, "cleaning_log.csv")
if not os.path.exists(IN_FILE):
    raise FileNotFoundError("ไม่พบ clean_output/clean_step2.csv ให้รัน clean_step2_missing.ipynb ให้สำเร็จก่อน")

df = pd.read_csv(IN_FILE).sort_values(["country", "year"]).reset_index(drop=True)
before_all = df.copy()

log_old = pd.read_csv(LOG_FILE)
log_old = log_old[~log_old["step"].str.startswith("3-")]   # รันซ้ำได้ไม่ซ้ำบรรทัด
log_rows = []
def log_step(action, before, after, reason):
    log_rows.append({"step": "3-outliers", "action": action,
                     "rows_before": before.shape[0], "rows_after": after.shape[0],
                     "cols_before": before.shape[1], "cols_after": after.shape[1], "reason": reason})
    print(f"✔ {action}: แถว {before.shape[0]:,} → {after.shape[0]:,} | คอลัมน์ {before.shape[1]} → {after.shape[1]}")

report = []   # ทุกค่าผิดปกติที่เจอ จะถูกจดลงที่นี่ แล้วบันทึกเป็น outlier_report.csv
def add_report(rows, check, column, decision, reason):
    for _, r in rows.iterrows():
        report.append({"country": r["country"], "year": r["year"], "check": check, "column": column,
                       "value": r[column], "decision": decision, "reason": reason})

print("ข้อมูลจากขั้นที่ 2:", df.shape, "| ประเทศ:", df["country"].nunique())

## 3.2 กฎเชิงโดเมน: หาค่าที่ "เป็นไปไม่ได้"
ก่อนใช้สถิติ ให้ใช้ความรู้เรื่องข้อมูลก่อน เพราะตรงไปตรงมาที่สุด

| กฎ | เหตุผล |
|---|---|
| สัดส่วน (%) ต้องอยู่ระหว่าง 0–100 | สัดส่วนเกิน 100% หรือติดลบเป็นไปไม่ได้ |
| ปริมาณไฟฟ้า/พลังงาน/ประชากร ต้อง ≥ 0 | ผลิตไฟฟ้าติดลบไม่ได้ (ยกเว้น `net_elec_imports` ที่ติดลบได้ = ส่งออกไฟฟ้า) |
| ความเข้มคาร์บอน 0–1,200 gCO₂/kWh | โรงไฟฟ้าถ่านหินที่สกปรกที่สุดปล่อยราว 1,000–1,100 |
| ไฟฟ้าส่วนย่อยต้องไม่เกินไฟฟ้าทั้งหมด | เช่น ไฟฟ้าจากถ่านหิน > ไฟฟ้าทั้งประเทศ เป็นไปไม่ได้ |

In [ ]:
SHARE_COLS = [c for c in df.columns if "share_elec" in c]
PARTS = [c for c in df.columns if c.endswith("_electricity") and c != "per_capita_electricity"]  # ไฟฟ้ารายเชื้อเพลิง (TWh)
AMOUNT_COLS = PARTS + ["electricity_generation", "electricity_demand", "per_capita_electricity", "population",
                       "primary_energy_consumption", "energy_per_capita", "greenhouse_gas_emissions", "gdp"]

rules = {
    "สัดส่วน < 0 หรือ > 100":       ((df[SHARE_COLS] < 0) | (df[SHARE_COLS] > 100.01)).any(axis=1),
    "ปริมาณติดลบ":                  (df[AMOUNT_COLS] < 0).any(axis=1),
    "ความเข้มคาร์บอนนอกช่วง 0-1200": ~df["carbon_intensity_elec"].between(0, 1200) & df["carbon_intensity_elec"].notna(),
    "ส่วนย่อย > ไฟฟ้าทั้งหมด":        (df[PARTS].gt(df["electricity_generation"] + 0.01, axis=0)).any(axis=1),
}
rule_result = pd.DataFrame({"กฎ": list(rules), "แถวที่ผิดกฎ": [int(m.sum()) for m in rules.values()]})
rule_result["ผล"] = np.where(rule_result["แถวที่ผิดกฎ"] == 0, "✅ ผ่าน", "❌ ต้องแก้")
rule_result

In [ ]:
# ถ้าผิดกฎ: ค่านั้นเชื่อไม่ได้ → เปลี่ยนเป็นค่าว่าง (ไม่เดาค่าใหม่) และจดรายงาน
before = df.copy()
n_fix = 0
for c in SHARE_COLS:
    bad = (df[c] < 0) | (df[c] > 100.01)
    add_report(df[bad], "domain_rule", c, "set_missing", "สัดส่วนนอกช่วง 0-100"); df.loc[bad, c] = np.nan; n_fix += bad.sum()
for c in AMOUNT_COLS:
    bad = df[c] < 0
    add_report(df[bad], "domain_rule", c, "set_missing", "ค่าติดลบที่เป็นไปไม่ได้"); df.loc[bad, c] = np.nan; n_fix += bad.sum()
log_step(f"ตรวจกฎเชิงโดเมน (แก้ {n_fix} ช่อง)", before, df, "ค่าที่เป็นไปไม่ได้ทางกายภาพ")

> **อ่านผล:** ข้อมูลชุดนี้ผ่านกฎทุกข้อ (เพราะเราจัดการ 0 ปลอมไปแล้วในขั้นที่ 2) แต่ต้องมีหลักฐานว่า **ได้ตรวจแล้ว** ไม่ใช่แค่สมมติว่าถูก

## 3.3 ตรวจด้วยสถิติ: IQR และ Z-score
| วิธี | สูตร | เหมาะกับ |
|---|---|---|
| **IQR** | ผิดปกติถ้า < Q1 − 1.5×IQR หรือ > Q3 + 1.5×IQR | ข้อมูล **เบ้** (ไม่สมมาตร) เพราะใช้ควอไทล์ ไม่ถูกค่าสุดโต่งดึง |
| **Z-score** | ผิดปกติถ้า \|z\| > 3 เมื่อ z = (x − mean) / std | ข้อมูลที่กระจายแบบ **ระฆังคว่ำ (ปกติ)** |

ตัวแปรหลักที่ตรวจคือ **`per_capita_electricity` (ไฟฟ้าต่อคน)** เพราะใช้เทียบประเทศได้ยุติธรรม
ตรวจ **แยกทีละปี** เพราะการใช้ไฟของโลกเพิ่มขึ้นทุกปี ถ้ารวมทุกปี ประเทศในปีหลัง ๆ จะดูสูงผิดปกติทั้งหมด

In [ ]:
# ดูรูปร่างการกระจายก่อนเลือกวิธี
LATEST = 2021          # ปีล่าสุดที่ข้อมูลครบ (2022 ยังไม่ครบ)
x = df.loc[df["year"] == LATEST, "per_capita_electricity"].dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].hist(x, bins=40); axes[0].set_title(f"Histogram ({LATEST})"); axes[0].set_xlabel("kWh per person")
axes[1].boxplot(x, vert=False); axes[1].set_title("Boxplot (linear scale)"); axes[1].set_xlabel("kWh per person")
axes[2].boxplot(np.log10(x[x > 0]), vert=False); axes[2].set_title("Boxplot (log10 scale)"); axes[2].set_xlabel("log10(kWh per person)")
plt.tight_layout(); plt.show()

print(f"mean = {x.mean():,.0f} | median = {x.median():,.0f} | skewness = {x.skew():.2f}")

> **อ่านผล:** ข้อมูล **เบ้ขวามาก** (skewness > 1, mean สูงกว่า median เกือบเท่าตัว) ประเทศส่วนใหญ่ใช้ไฟน้อย มีไม่กี่ประเทศที่ใช้มาก
> - Z-score จึง **ไม่เหมาะ** เพราะค่าสุดโต่งดึง mean และ std ให้สูงตาม ทำให้หาค่าผิดปกติไม่เจอ
> - **เลือก IQR** เป็นวิธีหลัก และแสดง Z-score เทียบไว้ให้เห็นความต่าง

In [ ]:
def iqr_flags(s, k=1.5):
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr), q1 - k * iqr, q3 + k * iqr

def z_flags(s, t=3):
    return ((s - s.mean()) / s.std()).abs() > t

col = "per_capita_electricity"
flag_iqr, lo, hi = iqr_flags(x)
print(f"ปี {LATEST}: ขอบเขต IQR = [{lo:,.0f}, {hi:,.0f}] kWh/คน")
print("IQR พบ    :", flag_iqr.sum(), "ประเทศ")
print("Z-score พบ:", z_flags(x).sum(), "ประเทศ  ← น้อยกว่า เพราะ std ถูกค่าสุดโต่งดึงให้กว้าง")

out = df.loc[x[flag_iqr].index, ["country", col, "renewables_share_elec", "fossil_share_elec"]].sort_values(col, ascending=False)
out

### ตรวจด้วยเหตุผลเชิงธุรกิจ: ผิดจริงหรือสูงจริง?
| ประเทศ | ทำไมใช้ไฟต่อคนสูง | ตัดสิน |
|---|---|---|
| ไอซ์แลนด์, นอร์เวย์ | ไฟฟ้าพลังน้ำ/ความร้อนใต้พิภพราคาถูก ดึงอุตสาหกรรมถลุงอะลูมิเนียมที่ใช้ไฟมหาศาล | **สูงจริง** |
| กาตาร์, คูเวต, บาห์เรน, UAE | อากาศร้อนจัด ใช้แอร์ทั้งปี + โรงกลั่นน้ำจืดจากน้ำทะเล + ค่าไฟถูก | **สูงจริง** |
| แคนาดา, สวีเดน, ฟินแลนด์ | อากาศหนาว ใช้ไฟทำความร้อน + อุตสาหกรรมหนัก | **สูงจริง** |

ทุกประเทศอธิบายได้ด้วยเหตุผลจริง **จึงเก็บไว้ทั้งหมด** การลบทิ้งจะทำให้แดชบอร์ดบิดเบือนความจริง
แต่เพิ่มคอลัมน์ธง `is_outlier_pc_elec` ไว้ แดชบอร์ดจะได้ใช้ **มัธยฐาน** และ **สเกล log** เมื่อต้องแสดงภาพรวม

In [ ]:
# ติดธงทุกปี (คำนวณ IQR แยกทีละปี)
def flag_by_year(data, col):
    return data.groupby("year")[col].transform(lambda s: iqr_flags(s.dropna())[0].reindex(s.index, fill_value=False))

before = df.copy()
df["is_outlier_pc_elec"] = flag_by_year(df, "per_capita_electricity").astype(bool)
add_report(df[df["is_outlier_pc_elec"]], "IQR_by_year", "per_capita_electricity", "keep_flag",
           "สูงจริง: อุตสาหกรรมใช้ไฟมาก/อากาศสุดขั้ว/ค่าไฟถูก")
log_step("เพิ่มธง is_outlier_pc_elec (IQR แยกรายปี) และเก็บค่าไว้", before, df,
         "ค่าสูงอธิบายได้ด้วยเหตุผลจริง ลบทิ้งจะบิดเบือนความจริง")
print("แถวที่ติดธง:", df["is_outlier_pc_elec"].sum(), "| ประเทศ:", df.loc[df["is_outlier_pc_elec"], "country"].nunique())

### ข้อควรระวัง: ตรวจตัวแปรผิดตัว = ได้ข้อสรุปผิด
ลองใช้ IQR กับ `electricity_generation` (ไฟฟ้ารวมทั้งประเทศ) ดู

In [ ]:
g = df.loc[df["year"] == LATEST, "electricity_generation"].dropna()
fg, _, _ = iqr_flags(g)
print("IQR บน electricity_generation พบ", fg.sum(), "ประเทศ เช่น:")
print(df.loc[g[fg].nlargest(8).index, "country"].tolist())

> **อ่านผล:** ได้จีน สหรัฐฯ อินเดีย ฯลฯ ซึ่ง **ไม่ใช่ความผิดปกติ** แค่เป็นประเทศใหญ่ ไฟฟ้ารวมจึงสูงตามขนาดประชากร
> บทเรียน: เวลาเทียบประเทศ ต้องใช้ตัวแปร **ต่อหัว** หรือ **สัดส่วน %** ไม่ใช่ยอดรวม ข้อนี้นำไปใช้ตอนออกแบบแดชบอร์ด

## 3.4 ตรวจการกระโดดตามเวลา (Year-over-year jump)
ค่าที่ผิดบ่อยที่สุดในข้อมูลรายปีคือ **ค่าที่เปลี่ยนจากปีก่อนแรงผิดปกติ**
เกณฑ์: ไฟฟ้าที่ผลิต **เปลี่ยนเกิน ±50% ในปีเดียว** (ดูเฉพาะประเทศที่ผลิต > 1 TWh เพราะประเทศเล็กมากเปลี่ยนนิดเดียวก็เป็นหลายสิบ %)

In [ ]:
gen_yoy = df.groupby("country")["electricity_generation"].pct_change() * 100   # % เปลี่ยนจากปีก่อน
jumps = df[(gen_yoy.abs() > 50) & (df["electricity_generation"] > 1)].assign(gen_yoy_pct=gen_yoy)
jumps[["country", "year", "electricity_generation", "gen_yoy_pct"]].round(1)

แต่ละรายการต้อง **หาเหตุผลรองรับ** ถ้ามีเหตุการณ์จริงอธิบายได้ ให้เก็บไว้ ถ้าไม่มี ให้สงสัยว่าเป็นข้อมูลผิด

| ประเทศ (ปี) | เหตุการณ์ที่อธิบายได้ |
|---|---|
| ลาว (2010, 2011, 2016) | เปิดเขื่อนน้ำเทิน 2 (2010) และโรงไฟฟ้าหงสา (2016) ส่วนใหญ่ขายไฟให้ไทย |
| ลิทัวเนีย (2010) | ปิดโรงไฟฟ้านิวเคลียร์ Ignalina ปลายปี 2009 (เป็นเงื่อนไขการเข้าร่วม EU) |
| ลักเซมเบิร์ก (2002) | เปิดโรงไฟฟ้าก๊าซขนาดใหญ่ (Twinerg) |
| บอตสวานา (2014) | เปิดโรงไฟฟ้าถ่านหิน Morupule B |
| มอลตา (2017) | เปิดโรงไฟฟ้าก๊าซแห่งใหม่ (Delimara) |
| แอลเบเนีย, ภูฏาน, อุรุกวัย ฯลฯ | ประเทศเล็กที่พึ่งพลังน้ำ ผันผวนตามปริมาณฝน หรือเปิดเขื่อนใหม่ |
| **โคโซโว (2022)** | **ไม่พบเหตุการณ์รองรับ** ไฟฟ้าถ่านหินเพิ่ม 57% ในปีเดียว ทั้งที่ไม่ได้เปิดโรงไฟฟ้าใหม่ และปี 2022 เป็นข้อมูลเบื้องต้นที่ยังไม่ครบ |

> **ตัดสิน:** เก็บทุกรายการที่อธิบายได้ ส่วน **โคโซโว 2022 เปลี่ยนเป็นค่าว่าง** เพราะน่าสงสัยและไม่มีหลักฐานรองรับ (ถ้าภายหลังพบหลักฐาน แก้ได้ที่ตัวแปร `SUSPECT`)

In [ ]:
SUSPECT = [("Kosovo", 2022)]          # (ประเทศ, ปี) ที่ตัดสินว่าข้อมูลน่าสงสัย
ELEC_COLS = [c for c in df.columns
             if c.endswith("_electricity") or "share_elec" in c
             or c in ["electricity_generation", "electricity_demand", "net_elec_imports",
                      "per_capita_electricity", "carbon_intensity_elec"]]

explained = jumps[~jumps.set_index(["country", "year"]).index.isin(SUSPECT)]
add_report(explained, "yoy_jump_gt50pct", "electricity_generation", "keep", "มีเหตุการณ์จริงรองรับ (เปิด/ปิดโรงไฟฟ้า, ปริมาณฝน)")

before = df.copy()
for ctry, yr in SUSPECT:
    m = (df["country"] == ctry) & (df["year"] == yr)
    add_report(df[m], "yoy_jump_gt50pct", "electricity_generation", "set_missing",
               "กระโดด >50% ไม่มีเหตุการณ์รองรับ + ปี 2022 ข้อมูลเบื้องต้น")
    df.loc[m, ELEC_COLS] = np.nan
log_step("เปลี่ยนค่าไฟฟ้าที่กระโดดผิดปกติโดยไม่มีเหตุผลรองรับเป็นค่าว่าง", before, df,
         "ค่าน่าสงสัย: " + ", ".join(f"{c} {y}" for c, y in SUSPECT))

df[df["country"] == "Kosovo"][["year", "electricity_generation", "coal_electricity"]].tail(4)

In [ ]:
# ประชากรก็ตรวจแบบเดียวกัน (เกณฑ์ ±15%/ปี เพราะประชากรเปลี่ยนช้ากว่าไฟฟ้ามาก)
pop_yoy = df.groupby("country")["population"].pct_change() * 100
pop_jumps = df.loc[pop_yoy.abs() > 15, ["country", "year", "population"]].assign(pop_yoy_pct=pop_yoy.round(1))
add_report(pop_jumps, "yoy_jump_gt15pct", "population", "keep", "แรงงานต่างชาติย้ายเข้าจำนวนมากช่วงเศรษฐกิจบูม")
pop_jumps

> **อ่านผล:** กาตาร์และ UAE ประชากรเพิ่มเกือบ 20% ต่อปีช่วง 2006–2008 จากแรงงานต่างชาติที่ย้ายเข้ามาช่วงก่อสร้างบูม เป็นเรื่องจริง **เก็บไว้**

## 3.5 ความแม่นยำของประเทศเล็กมาก
ประเทศที่ผลิตไฟฟ้าน้อยมาก (เช่น หมู่เกาะฟอล์กแลนด์ ราว 0.02 TWh) ตัวเลขในแหล่งข้อมูลถูก **ปัดทศนิยม** ไว้แค่ 2 ตำแหน่ง
พอนำมาหารกันเพื่อคำนวณสัดส่วนหรือความเข้มคาร์บอน ผลจะเพี้ยนง่าย เช่น ความเข้มคาร์บอนออกมาเป็นเลขกลม ๆ อย่าง 1,000 พอดี

In [ ]:
odd = df[df["carbon_intensity_elec"] >= 1000][["country", "year", "electricity_generation", "carbon_intensity_elec"]]
print("ความเข้มคาร์บอน >= 1,000 (สูงเกินโรงไฟฟ้าน้ำมันทั่วไปที่ราว 650-750):")
display(odd.groupby("country").agg(ปี=("year", "count"), ผลิตไฟฟ้า_TWh=("electricity_generation", "median"),
                                   ความเข้มคาร์บอน=("carbon_intensity_elec", "median")))

SMALL_TWH = 0.1
before = df.copy()
df["low_precision"] = df["electricity_generation"] < SMALL_TWH
add_report(df[df["low_precision"] & (df["carbon_intensity_elec"] >= 1000)], "rounding_small_country",
           "carbon_intensity_elec", "keep_flag", f"ผลิตไฟฟ้า < {SMALL_TWH} TWh ตัวเลขถูกปัดจนเพี้ยน")
log_step(f"เพิ่มธง low_precision (ผลิตไฟฟ้า < {SMALL_TWH} TWh)", before, df,
         "ประเทศเล็กมาก ตัวเลขถูกปัด สัดส่วน/ความเข้มคาร์บอนอาจเพี้ยน")
print("แถวที่ติดธง low_precision:", df["low_precision"].sum(), "| ประเทศ:", df.loc[df["low_precision"], "country"].nunique())

> **ตัดสิน:** เก็บไว้ แต่ติดธง `low_precision` ให้แดชบอร์ดแสดงหมายเหตุ หรือกรองออกได้เวลาจัดอันดับ (ไม่อย่างนั้นอันดับ 1 "ประเทศปล่อยคาร์บอนต่อหน่วยไฟสูงสุด" จะเป็นเกาะเล็ก ๆ ที่ตัวเลขเพี้ยน)

## 3.6 หลักฐานก่อน–หลัง และผลกระทบต่อสถิติ
เราไม่ได้ลบค่าสูงจริงออก จึงต้องแสดงว่า **ถ้าลบทิ้ง สถิติจะเพี้ยนแค่ไหน** เพื่อยืนยันว่าการเก็บไว้แล้วใช้มัธยฐานเป็นทางที่ถูก

In [ ]:
y = df[df["year"] == LATEST]
with_out = y["per_capita_electricity"].dropna()
without_out = y.loc[~y["is_outlier_pc_elec"], "per_capita_electricity"].dropna()

impact = pd.DataFrame({
    "รวมค่าสูงจริง (ที่เราเลือก)": [len(with_out), with_out.mean(), with_out.median(), with_out.std()],
    "ถ้าลบค่าสูงทิ้ง": [len(without_out), without_out.mean(), without_out.median(), without_out.std()],
}, index=["จำนวนประเทศ", "mean (kWh/คน)", "median (kWh/คน)", "std"]).round(0)
impact["เปลี่ยนไป (%)"] = ((impact.iloc[:, 1] / impact.iloc[:, 0] - 1) * 100).round(1)
impact

> **อ่านผล:** ถ้าลบค่าสูงทิ้ง **mean จะลดลงมาก** แต่ **median เปลี่ยนน้อย**
> สรุป: ค่าสุดโต่งกระทบ mean รุนแรง จึงควรรายงาน **มัธยฐาน** เป็นค่ากลางหลักบนแดชบอร์ด แล้วเก็บข้อมูลจริงไว้ครบ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (label, data) in zip(axes, [("before step 3", before_all), ("after step 3", df)]):
    d = data[data["year"] == LATEST]
    ax.boxplot([np.log10(d["per_capita_electricity"].dropna().clip(lower=1)),
                np.log10(d["electricity_generation"].dropna().clip(lower=0.001))], vert=False)
    ax.set_yticks([1, 2], ["per_capita_electricity", "electricity_generation"])
    ax.set_xlabel("log10 scale"); ax.set_title(f"Boxplot {LATEST} ({label})")
plt.tight_layout(); plt.show()
print("หมายเหตุ: กราฟก่อน–หลังเกือบเหมือนกัน เพราะเราเลือก 'เก็บและติดธง' ไม่ใช่ลบ ยกเว้นโคโซโว 2022 ที่ไม่อยู่ในปี 2021")

In [ ]:
# ตรวจกฎเชิงโดเมนซ้ำหลังทำ (ต้องผ่านทั้งหมด)
recheck = {
    "สัดส่วน 0-100": (((df[SHARE_COLS] < 0) | (df[SHARE_COLS] > 100.01)).sum().sum() == 0),
    "ไม่มีค่าติดลบ": ((df[AMOUNT_COLS] < 0).sum().sum() == 0),
    "ความเข้มคาร์บอน 0-1200": df["carbon_intensity_elec"].dropna().between(0, 1200).all(),
}
for k, v in recheck.items():
    print("✅" if v else "❌", k)

rep = pd.DataFrame(report)
print("\nสรุปรายงานค่าผิดปกติ (จำนวนแถว):")
rep.groupby(["check", "decision"]).size().to_frame("แถว")

## 3.7 บันทึกไฟล์

In [ ]:
df.to_csv(os.path.join(OUT_DIR, "clean_step3.csv"), index=False, encoding="utf-8-sig")
rep.to_csv(os.path.join(OUT_DIR, "outlier_report.csv"), index=False, encoding="utf-8-sig")
pd.concat([log_old, pd.DataFrame(log_rows)]).to_csv(LOG_FILE, index=False, encoding="utf-8-sig")

print("ข้อมูลหลังขั้นที่ 3:", df.shape)
print("คอลัมน์ธงใหม่: is_outlier_pc_elec, low_precision")
print("\nบันทึกแล้ว:")
for f in sorted(os.listdir(OUT_DIR)):
    print("  -", f, f"({os.path.getsize(os.path.join(OUT_DIR, f))/1024:,.0f} KB)")

In [ ]:
pd.read_csv(LOG_FILE)

## ✅ เช็กลิสต์ก่อนไปขั้นที่ 4
- [ ] รันครบไม่มี error และการตรวจซ้ำข้อ 3.6 ขึ้น ✅ ทั้ง 3 ข้อ
- [ ] มีไฟล์ใหม่ `clean_step3.csv` และ `outlier_report.csv` (เปิดใน Excel ดูได้ว่าแต่ละค่าตัดสินว่าอะไร เพราะอะไร)
- [ ] อธิบายได้ว่า **ทำไมเลือก IQR แทน Z-score** (ข้อมูลเบ้ขวา)
- [ ] อธิบายได้ว่า **ทำไมเก็บไอซ์แลนด์ไว้ แต่เปลี่ยนโคโซโว 2022 เป็นค่าว่าง**

**ประโยคสรุปไว้ตอบกรรมการ:** "เราตรวจค่าผิดปกติ 4 ชั้น คือกฎโดเมน, IQR แยกรายปี, การกระโดดตามเวลา และความแม่นยำของประเทศเล็ก แล้วแยกให้ได้ว่า 'ผิดจริง' หรือ 'สูงจริง' ค่าสูงจริงเก็บไว้และติดธง ส่วนค่าที่ไม่มีเหตุผลรองรับเปลี่ยนเป็นค่าว่าง ทุกการตัดสินอยู่ใน outlier_report.csv และเพราะข้อมูลเบ้ แดชบอร์ดจึงใช้มัธยฐานเป็นค่ากลางหลัก"

**ขั้นถัดไป (ไฟล์ที่ 4):** ทำรูปแบบให้เป็นมาตรฐาน (หน่วย ชื่อคอลัมน์ ทศนิยม) ตรวจว่าตัวเลขสอดคล้องกัน (เช่น สัดส่วนเชื้อเพลิงรวมได้ 100%) และติดป้ายปี 2022 ว่าข้อมูลไม่ครบ